# Direct Spherical Context Encoder–Decoder

教授方案：

\[
x_{\text{context}}\rightarrow z_{\text{spherical}}\rightarrow\hat y_{\text{center word}}
\]

不使用预训练 Word2Vec。输入是上下文，输出是中心词，因此这是 **encoder–decoder with a spherical bottleneck**，不是传统 autoencoder。

`nn.EmbeddingBag` 等价于对 context multi-hot vector 做线性变换 \(Wx_{\text{context}}\)，但无需显式构造30,000维稠密向量。该矩阵从零开始训练，并不是外部 Word2Vec。


In [1]:
import random, urllib.request, zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from IPython.display import display


In [2]:
SEED = 42
VOCAB_SIZE = 30_000
WINDOW_RADIUS = 5
MIN_COUNT = 5
BATCH_SIZE = 256
EPOCHS = 50
LEARNING_RATE = 2e-3
CONTEXT_HIDDEN_DIM = 256
ENCODER_HIDDEN_DIM = 128
LATENT_DIM = 32
SAMPLE_ALPHA = 0.15
BETA_RIESZ = 1e-3
RIESZ_SUBSAMPLE = 256
MAX_TOKENS = 3_000_000

DATA_DIR = Path("./data")
TEXT8_PATH = DATA_DIR / "text8"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)


Device: cuda


## 1. Load text8


In [3]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not TEXT8_PATH.exists():
    zip_path = DATA_DIR / "text8.zip"
    urllib.request.urlretrieve(
        "http://mattmahoney.net/dc/text8.zip",
        zip_path,
    )
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(DATA_DIR)

tokens = TEXT8_PATH.read_text(
    encoding="utf-8"
).split()

if MAX_TOKENS is not None:
    tokens = tokens[:MAX_TOKENS]

print("Tokens:", len(tokens))


Tokens: 3000000


## 2. Vocabulary


In [4]:
counter = Counter(tokens)

words = [
    word
    for word, count in counter.most_common(VOCAB_SIZE - 2)
    if count >= MIN_COUNT
]

idx_to_word = ["<pad>", "<unk>"] + words
word_to_idx = {
    word: index
    for index, word in enumerate(idx_to_word)
}

PAD_ID = word_to_idx["<pad>"]
UNK_ID = word_to_idx["<unk>"]

encoded_tokens = np.asarray(
    [word_to_idx.get(word, UNK_ID) for word in tokens],
    dtype=np.int64,
)

ACTUAL_VOCAB_SIZE = len(idx_to_word)

print("Vocabulary size:", ACTUAL_VOCAB_SIZE)
print("UNK ratio:", float(np.mean(encoded_tokens == UNK_ID)))


Vocabulary size: 27323
UNK ratio: 0.036664


## 3. Context → center word dataset

窗口半径为5：

```text
5 left words + center word + 5 right words
```

模型输入10个上下文词，输出中心词。


In [5]:
class ContextCenterDataset(Dataset):
    def __init__(self, token_ids, window_radius=5):
        self.token_ids = token_ids
        self.window_radius = int(window_radius)

    def __len__(self):
        return len(self.token_ids) - 2 * self.window_radius

    def __getitem__(self, index):
        center_position = index + self.window_radius

        left = self.token_ids[
            center_position - self.window_radius:
            center_position
        ]

        right = self.token_ids[
            center_position + 1:
            center_position + self.window_radius + 1
        ]

        context = np.concatenate([left, right])
        center = self.token_ids[center_position]

        return (
            torch.tensor(context, dtype=torch.long),
            torch.tensor(center, dtype=torch.long),
        )


split = int(0.90 * len(encoded_tokens))

train_dataset = ContextCenterDataset(
    encoded_tokens[:split],
    WINDOW_RADIUS,
)

valid_dataset = ContextCenterDataset(
    encoded_tokens[split - 2 * WINDOW_RADIUS:],
    WINDOW_RADIUS,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print("Train:", len(train_dataset))
print("Validation:", len(valid_dataset))


Train: 2699990
Validation: 300000


## 4. Direct spherical model


In [6]:
def riesz_energy(points, subsample=256, s=1.0):
    if points.size(0) < 2:
        return torch.zeros(
            (),
            device=points.device,
            dtype=points.dtype,
        )

    if subsample is not None and points.size(0) > subsample:
        indices = torch.randperm(
            points.size(0),
            device=points.device,
        )[:subsample]
        points = points[indices]

    distances = torch.cdist(points, points)

    mask = torch.triu(
        torch.ones_like(distances, dtype=torch.bool),
        diagonal=1,
    )

    valid = distances[mask]

    return torch.mean(
        1.0 / torch.clamp(valid, min=1e-4).pow(s)
    )


class DirectSphericalContextModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        context_hidden_dim=256,
        encoder_hidden_dim=128,
        latent_dim=32,
        sample_alpha=0.15,
    ):
        super().__init__()

        self.sample_alpha = float(sample_alpha)

        self.context_projection = nn.EmbeddingBag(
            vocab_size,
            context_hidden_dim,
            mode="sum",
        )

        self.encoder = nn.Sequential(
            nn.Linear(context_hidden_dim, encoder_hidden_dim),
            nn.ReLU(),
            nn.Linear(encoder_hidden_dim, latent_dim),
        )

        self.decoder = nn.Linear(
            latent_dim,
            vocab_size,
        )

    def encode_context(self, context_ids):
        batch_size, context_length = context_ids.shape

        flattened = context_ids.reshape(-1)

        offsets = torch.arange(
            0,
            batch_size * context_length,
            context_length,
            device=context_ids.device,
        )

        context_hidden = self.context_projection(
            flattened,
            offsets,
        )

        raw_direction = self.encoder(context_hidden)

        return F.normalize(
            raw_direction,
            p=2,
            dim=-1,
            eps=1e-8,
        )

    def sample_on_sphere(self, mean_direction):
        ambient_noise = torch.randn_like(mean_direction)

        tangent_noise = (
            ambient_noise
            - (
                ambient_noise * mean_direction
            ).sum(dim=-1, keepdim=True)
            * mean_direction
        )

        tangent_noise = F.normalize(
            tangent_noise,
            p=2,
            dim=-1,
            eps=1e-8,
        )

        theta = self.sample_alpha * torch.rand(
            mean_direction.size(0),
            1,
            device=mean_direction.device,
        )

        z = (
            mean_direction * torch.cos(theta)
            + tangent_noise * torch.sin(theta)
        )

        return F.normalize(
            z,
            p=2,
            dim=-1,
            eps=1e-8,
        )

    def forward(self, context_ids, sample=True):
        mean_direction = self.encode_context(context_ids)

        z = (
            self.sample_on_sphere(mean_direction)
            if sample
            else mean_direction
        )

        logits = self.decoder(z)

        return {
            "logits": logits,
            "mean_direction": mean_direction,
            "z": z,
        }


model = DirectSphericalContextModel(
    vocab_size=ACTUAL_VOCAB_SIZE,
    context_hidden_dim=CONTEXT_HIDDEN_DIM,
    encoder_hidden_dim=ENCODER_HIDDEN_DIM,
    latent_dim=LATENT_DIM,
    sample_alpha=SAMPLE_ALPHA,
).to(DEVICE)

print(model)


DirectSphericalContextModel(
  (context_projection): EmbeddingBag(27323, 256, mode='sum')
  (encoder): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=32, bias=True)
  )
  (decoder): Linear(in_features=32, out_features=27323, bias=True)
)


## 5. Loss

\[
\mathcal L =
\mathcal L_{\text{center-word classification}}
+
\beta\mathcal L_{\text{Riesz}}
\]


In [7]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

history = []

for epoch in range(EPOCHS):
    model.train()

    totals = defaultdict(float)
    steps = 0

    for context_ids, center_ids in train_loader:
        context_ids = context_ids.to(
            DEVICE,
            non_blocking=True,
        )

        center_ids = center_ids.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        output = model(
            context_ids,
            sample=True,
        )

        center_loss = F.cross_entropy(
            output["logits"],
            center_ids,
        )

        riesz_loss = riesz_energy(
            output["mean_direction"],
            subsample=RIESZ_SUBSAMPLE,
        )

        loss = (
            center_loss
            + BETA_RIESZ * riesz_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        top1 = (
            output["logits"]
            .argmax(dim=-1)
            .eq(center_ids)
            .float()
            .mean()
        )

        totals["loss"] += float(loss.detach().cpu())
        totals["center"] += float(center_loss.detach().cpu())
        totals["riesz"] += float(riesz_loss.detach().cpu())
        totals["top1"] += float(top1.detach().cpu())
        steps += 1

    row = {
        "epoch": epoch + 1,
        "loss": totals["loss"] / steps,
        "center_loss": totals["center"] / steps,
        "riesz": totals["riesz"] / steps,
        "train_top1": totals["top1"] / steps,
    }

    history.append(row)
    print(row)

history = pd.DataFrame(history)
display(history)


{'epoch': 1, 'loss': 6.68824777994674, 'center_loss': 6.6868586338582485, 'riesz': 1.3891459440375726, 'train_top1': 0.10827855055093358}
{'epoch': 2, 'loss': 6.267482053954375, 'center_loss': 6.266685772385204, 'riesz': 0.7962831495677825, 'train_top1': 0.13200728974390835}
{'epoch': 3, 'loss': 6.068026468834516, 'center_loss': 6.067266074381609, 'riesz': 0.760393303492018, 'train_top1': 0.14354395121265343}
{'epoch': 4, 'loss': 5.922989739075966, 'center_loss': 5.922245093767277, 'riesz': 0.7446442670230787, 'train_top1': 0.1518020014876131}
{'epoch': 5, 'loss': 5.805171401825993, 'center_loss': 5.80443529627431, 'riesz': 0.7361040534798631, 'train_top1': 0.1590259172551449}
{'epoch': 6, 'loss': 5.7052321872564695, 'center_loss': 5.704501178125143, 'riesz': 0.7310094119440699, 'train_top1': 0.16563974679607096}
{'epoch': 7, 'loss': 5.617873308974397, 'center_loss': 5.617144951404887, 'riesz': 0.7283574561377866, 'train_top1': 0.17189013654500532}
{'epoch': 8, 'loss': 5.54050996471516

,epoch,loss,center_loss,riesz,train_top1
0,1,6.688248,6.686859,1.389146,0.108279
1,2,6.267482,6.266686,0.796283,0.132007
2,3,6.068026,6.067266,0.760393,0.143544
3,4,5.922990,5.922245,0.744644,0.151802
4,5,5.805171,5.804435,0.736104,0.159026
5,6,5.705232,5.704501,0.731009,0.165640
6,7,5.617873,5.617145,0.728357,0.171890
7,8,5.540510,5.539783,0.726748,0.177676
8,9,5.470666,5.469940,0.726139,0.182933
9,10,5.407044,5.406318,0.726028,0.188147


## 6. Validation


In [8]:
def evaluate_model(model, loader):
    model.eval()

    totals = defaultdict(float)
    steps = 0

    with torch.no_grad():
        for context_ids, center_ids in loader:
            context_ids = context_ids.to(DEVICE)
            center_ids = center_ids.to(DEVICE)

            output = model(
                context_ids,
                sample=False,
            )

            loss = F.cross_entropy(
                output["logits"],
                center_ids,
            )

            top5 = torch.topk(
                output["logits"],
                k=5,
                dim=-1,
            ).indices

            totals["loss"] += float(loss.cpu())

            totals["top1"] += float(
                top5[:, 0]
                .eq(center_ids)
                .float()
                .mean()
                .cpu()
            )

            totals["top5"] += float(
                top5
                .eq(center_ids.unsqueeze(1))
                .any(dim=1)
                .float()
                .mean()
                .cpu()
            )

            totals["latent_norm"] += float(
                output["z"]
                .norm(dim=-1)
                .mean()
                .cpu()
            )

            steps += 1

    return {
        key: value / steps
        for key, value in totals.items()
    }


display(
    pd.DataFrame([
        evaluate_model(model, valid_loader)
    ])
)


,loss,top1,top5,latent_norm
0,7.486697,0.091826,0.209036,1.0


## 7. Inspect predictions


In [9]:
model.eval()

context_batch, center_batch = next(
    iter(valid_loader)
)

context_batch = context_batch[:12].to(DEVICE)
center_batch = center_batch[:12].to(DEVICE)

with torch.no_grad():
    output = model(
        context_batch,
        sample=False,
    )

    top5_ids = torch.topk(
        output["logits"],
        k=5,
        dim=-1,
    ).indices.cpu().numpy()

rows = []

for index in range(len(context_batch)):
    rows.append({
        "context": " ".join(
            idx_to_word[token_id]
            for token_id in context_batch[index].cpu().numpy()
        ),
        "target": idx_to_word[
            int(center_batch[index].cpu())
        ],
        "top5_predictions": [
            idx_to_word[token_id]
            for token_id in top5_ids[index]
        ],
    })

display(pd.DataFrame(rows))


,context,target,top5_predictions
0,the well informed is <unk> as an inspiration o...,offered,"[as, <unk>, art, writer, character]"
1,well informed is <unk> offered an inspiration ...,as,"[as, that, also, which, best]"
2,informed is <unk> offered as inspiration of th...,an,"[also, to, as, many, for]"
3,is <unk> offered as an of the baroque which ap...,inspiration,"[<unk>, variety, style, art, character]"
4,<unk> offered as an inspiration the baroque wh...,of,"[<unk>, new, was, him, greek]"
5,offered as an inspiration of baroque which app...,the,"[<unk>, the, variety, piece, was]"
6,as an inspiration of the which appeared howeve...,baroque,"[new, between, <unk>, was, story]"
7,an inspiration of the baroque appeared however...,which,"[new, period, style, <unk>, as]"
8,inspiration of the baroque which however a gen...,appeared,"[was, became, period, made, continued]"
9,of the baroque which appeared a generation lat...,however,"[in, on, with, at, was]"


## 8. Build word-level spherical representations

这里有一个重要区别：模型原始 latent 表示的是一个**上下文窗口**，不是单独的词。

为了得到每个词的球面表示，我们聚合“该词作为中心词时”的所有 context latent：

\[
e_w =
\operatorname{normalize}
\left(
\sum_{i:y_i=w}z_i
\right)
\]


In [10]:
def build_word_spherical_embeddings(
    model,
    loader,
    vocab_size,
    max_batches=None,
):
    model.eval()

    sums = np.zeros(
        (vocab_size, LATENT_DIM),
        dtype=np.float64,
    )

    counts = np.zeros(
        vocab_size,
        dtype=np.int64,
    )

    with torch.no_grad():
        for batch_index, (
            context_ids,
            center_ids,
        ) in enumerate(loader):
            if (
                max_batches is not None
                and batch_index >= max_batches
            ):
                break

            output = model(
                context_ids.to(DEVICE),
                sample=False,
            )

            latents = output["z"].cpu().numpy()
            centers = center_ids.numpy()

            for word_id in np.unique(centers):
                mask = centers == word_id
                sums[word_id] += latents[mask].sum(axis=0)
                counts[word_id] += int(mask.sum())

    embeddings = np.zeros_like(sums)
    valid = counts > 0

    norms = np.linalg.norm(
        sums[valid],
        axis=1,
        keepdims=True,
    )

    embeddings[valid] = (
        sums[valid]
        / np.clip(norms, 1e-12, None)
    )

    return embeddings, counts


word_spherical_embeddings, word_context_counts = (
    build_word_spherical_embeddings(
        model,
        train_loader,
        ACTUAL_VOCAB_SIZE,
    )
)

print(
    "Words represented:",
    int(np.sum(word_context_counts > 0)),
)


Words represented: 27181


## 9. Nearest neighbors


In [11]:
def nearest_words(query_word, top_k=10):
    if query_word not in word_to_idx:
        raise KeyError(query_word)

    query_id = word_to_idx[query_word]
    query = word_spherical_embeddings[query_id]

    if np.linalg.norm(query) < 1e-12:
        raise ValueError("No representation for this word.")

    similarities = (
        word_spherical_embeddings @ query
    )

    similarities[word_context_counts == 0] = -np.inf
    similarities[query_id] = -np.inf

    nearest_ids = np.argsort(
        -similarities
    )[:top_k]

    return pd.DataFrame({
        "word": [
            idx_to_word[index]
            for index in nearest_ids
        ],
        "cosine_similarity": similarities[nearest_ids],
        "context_count": word_context_counts[nearest_ids],
    })


for query_word in [
    "king",
    "queen",
    "france",
    "italy",
    "man",
    "woman",
]:
    if query_word in word_to_idx:
        print("Query:", query_word)
        display(
            nearest_words(
                query_word,
                top_k=10,
            )
        )


Query: king


,word,cosine_similarity,context_count
0,prince,0.893793,245
1,regent,0.888737,29
2,emperor,0.879003,535
3,grandson,0.878657,41
4,macedon,0.856865,53
5,queen,0.844880,286
6,son,0.844073,775
7,ruler,0.841181,114
8,empress,0.836727,24
9,sancho,0.829213,16


Query: queen


,word,cosine_similarity,context_count
0,lord,0.853423,344
1,king,0.844880,1304
2,consort,0.824991,14
3,prince,0.814062,245
4,bishop,0.797012,191
5,princess,0.793187,99
6,theresa,0.791983,7
7,bentinck,0.786843,5
8,lady,0.776479,68
9,toulouse,0.773817,28


Query: france


...

,word,cosine_similarity,context_count
0,france,0.903965,637
1,spain,0.897890,294
2,switzerland,0.882866,96
3,germany,0.882749,706
4,india,0.867642,447
5,prussia,0.863410,71
6,persia,0.862374,88
7,russia,0.856913,388
8,hungary,0.856630,154
9,thrace,0.856575,32


Query: man


,word,cosine_similarity,context_count
0,woman,0.916690,264
1,evil,0.860726,153
2,passion,0.857415,67
3,hero,0.850737,120
4,dream,0.849514,69
5,love,0.835266,363
6,forehead,0.832811,10
7,child,0.819378,226
8,sacrifice,0.812128,81
9,carefree,0.809947,6


Query: woman


,word,cosine_similarity,context_count
0,man,0.916690,959
1,child,0.876790,226
2,herself,0.816417,93
3,dione,0.800072,5
4,sacrifice,0.796507,81
5,girl,0.793964,109
6,curse,0.788774,27
7,hero,0.784618,120
8,villain,0.782311,13
9,wise,0.781710,47


## 10. Save model and embeddings


In [12]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "word_to_idx": word_to_idx,
        "idx_to_word": idx_to_word,
    },
    "direct_spherical_context_model.pt",
)

np.savez_compressed(
    "direct_spherical_word_embeddings.npz",
    embeddings=word_spherical_embeddings,
    counts=word_context_counts,
    words=np.asarray(
        idx_to_word,
        dtype=object,
    ),
)

print("Saved.")


Saved.
